# D4RL-faithful Ant: staged 250k nominal reproduction (binary NCE, target-entropy -8)

Long-run nominal reproduction on the **d4rl-faithful branch** (`crl/d4rl_ant.py`):
verbatim D4RL ant XML physics (RK4 / 0.02 / ctrl +-30 gear-1 / dt 0.1), U-maze,
**full 29-dim settled-state goals**, corrected adaptive entropy (target -8),
numerical guards, single actor, binary NCE. **No causal modification, no BC, no
repr-norm, no added collection noise.**

Staged to **250k env steps** with mandatory gates at **50k / 100k / 150k / 200k /
250k**. The run STOPS at 250k -- it does not continue to 1M automatically.

Conventions reused from `notebooks/antmaze_umaze_qualification.ipynb` (see section 3).
Colab tips: use a GPU runtime; on crash use Runtime > Disconnect and delete, then
rerun from the top with `RESUME=True`.

## 1. Configuration
Everything tunable lives here. The same `RUN_ID` names local paths, Drive paths, logs, and reports.

In [ ]:
# 1. Configuration -- single source of truth for the whole notebook.
import os, time

SEED             = 0
MAX_STEP         = 250_000                 # hard stop; do NOT auto-continue to 1M
GATES            = [50_000, 100_000, 150_000, 200_000, 250_000]
CKPT_EVERY       = 10_000                  # resumable latest.pkl cadence (atomic)
RESUME           = False                   # True after a disconnect
ALLOW_FRESH_START = False                  # explicit opt-in for step-0 starts
TAG              = ''                      # optional extra experiment tag

ENV              = 'd4rl_ant_umaze_gfull'  # faithful physics + full-state goal
GOAL_REPR        = 'gfull29'               # settled 29-dim commanded goal
ENTROPY_COEF     = None                    # None => adaptive alpha
TARGET_ENTROPY   = -8.0                    # verified direction (alpha sanity test)
NUM_ACTORS       = 1                       # single-process collection (documented)
REQUIRE_GPU      = True                    # abort if no accelerator

REPO_URL         = 'https://github.com/tingrui-huang/contrastive_rl.git'
BRANCH           = 'main'
COMMIT           = ''                      # ''=track origin/BRANCH head; pin a hash
                                           # here once this run is archived.

RUN_ID = (f'{ENV}_{GOAL_REPR}_te{int(abs(TARGET_ENTROPY))}_'
          f'{NUM_ACTORS}actor_s{SEED}_{MAX_STEP // 1000}k'
          + (f'_{TAG}' if TAG else ''))
DRIVE_BASE = '/content/drive/MyDrive/contrastive_rl_runs/d4rl_ant_nominal'
RUN_DIR    = f'{DRIVE_BASE}/{RUN_ID}'      # persistent (Drive)
LOCAL      = f'/content/scratch_{RUN_ID}'  # fast temporary scratch
REPO       = '/content/contrastive_rl'
SUBDIRS    = ['checkpoints', 'gates', 'attribution', 'gradient', 'logs',
              'summary', 'meta']           # (replay snapshots: not supported by
                                           #  the trainer -- buffer refills on
                                           #  resume; documented in section 8)
print('RUN_ID :', RUN_ID)
print('RUN_DIR:', RUN_DIR)
print('RESUME :', RESUME, '| ALLOW_FRESH_START:', ALLOW_FRESH_START)

## 2. Mount Google Drive
All persistent outputs live under `RUN_DIR`; `/content` is scratch only.

In [ ]:
# 2. Mount Drive + create the run directory tree + disk usage.
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(LOCAL, exist_ok=True)
for d in SUBDIRS:
    os.makedirs(f'{RUN_DIR}/{d}', exist_ok=True)
print('run tree:')
for d in SUBDIRS:
    print('  ', f'{RUN_DIR}/{d}')
print('\ndisk usage:')
!df -h /content | tail -1
!du -sh $RUN_DIR 2>/dev/null || echo '(new run dir)'

## 3. Repository notebook conventions reused
From `notebooks/antmaze_umaze_qualification.ipynb` (the repo's proven Colab flow):
- **dependency install**: the proven `--no-deps` recipe holding Colab's GPU
  jax/jaxlib/numpy fixed, `XLA_PYTHON_CLIENT_PREALLOCATE=false`, `MUJOCO_GL=egl`
  (cell 1 there, verbatim here in section 5);
- **clone + pinned checkout** of the GitHub fork (cell 2), extended with a
  dirty-tree guard so uncommitted work is never overwritten;
- **Drive mount** (cell 3) and **checkpoints written directly to a Drive run
  dir** (`crl.checkpoint` writes `latest.pkl` atomically: tmp + `os.replace`);
- **config cell + `train(cfg)` in-process** as the real training entry point
  (cells 4/7), with the same guard-railed `Config` asserts;
- **pre-training audit gate that stops the notebook on failure** (cell 5 ->
  our section 9 runs `scripts/verify_d4rl_ant.py`);
- **TensorBoard on `<ckpt_dir>/tb`** via tensorboardX (cell 6);
- **post-run report + verdict packaging** (cell 8 -> our sections 11/13/14).

Deviations (all deliberate, listed again in the final report): staged
5x50k `train()` calls with resume between stages (gates need hard checkpoint
boundaries), stdout tee to a Drive log file, rolling keep-3 checkpoints, and
`COMMIT=''` tracking `origin/main` until the run is archived.

## 4. Clone or update repository (never overwrites uncommitted work)

In [ ]:
# 4. Clone if absent; otherwise fetch + checkout safely.
import subprocess
os.chdir('/content')
if not os.path.exists(REPO):
    !git clone $REPO_URL $REPO
os.chdir(REPO)
dirty = subprocess.run(['git', 'status', '--porcelain'],
                       capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError('repository has uncommitted changes -- refusing to '
                       'checkout over them:\n' + dirty)
!git fetch -q origin
ref = COMMIT if COMMIT else f'origin/{BRANCH}'
!git checkout -q $ref
print('branch/ref :', ref)
!git log -1 --oneline
!git status --short --branch | head -3

## 5. Install dependencies (proven Colab recipe -- do not alter)

In [ ]:
# 5. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX.
# (verbatim proven-safe recipe from antmaze_umaze_qualification.ipynb cell 1)
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # do not grab all VRAM
os.environ["MUJOCO_GL"] = "egl"                         # headless render backend
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}",
        f"numpy=={numpy.__version__}"]
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps", "dm-haiku", "optax", "chex")            # jax libs: no dep changes
pip("jmp", "tabulate", "toolz", "etils", "tensorboardX",  # pure-python deps + TB
    "gymnasium-robotics", "mujoco", "imageio-ffmpeg", *hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

## 6. Environment and GPU verification (saved to Drive)

In [ ]:
# 6. Versions, devices, git commit, XML checksum -> meta/env_metadata.json.
import hashlib, json, platform, importlib.metadata as im
import jax, jaxlib, mujoco
os.chdir(REPO)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
xml_sha = hashlib.sha256(
    open('crl/assets/d4rl_ant.xml', 'rb').read()).hexdigest()
meta = {
    'run_id': RUN_ID, 'python': platform.python_version(),
    'jax': jax.__version__, 'jaxlib': jaxlib.__version__,
    'backend': jax.default_backend(),
    'devices': [str(d) for d in jax.devices()],
    'mujoco': mujoco.__version__,
    'gymnasium': im.version('gymnasium'),
    'git_commit': commit, 'd4rl_ant_xml_sha256': xml_sha,
    'config': {'env': ENV, 'goal': GOAL_REPR, 'seed': SEED,
               'max_step': MAX_STEP, 'gates': GATES,
               'entropy_coefficient': ENTROPY_COEF,
               'target_entropy': TARGET_ENTROPY, 'num_actors': NUM_ACTORS,
               'ckpt_every': CKPT_EVERY},
}
if REQUIRE_GPU and jax.default_backend() == 'cpu':
    raise RuntimeError('no accelerator -- Runtime > Change runtime type > GPU')
!nvidia-smi -L || true
json.dump(meta, open(f'{RUN_DIR}/meta/env_metadata.json', 'w'), indent=2)
print(json.dumps(meta, indent=1))

## 7. Persistent paths and run identity

In [ ]:
# 7. Record the run config on Drive; one RUN_ID everywhere.
CKPT_DIR = f'{RUN_DIR}/checkpoints'        # trainer writes here (atomic latest)
json.dump({'run_id': RUN_ID, 'run_dir': RUN_DIR, 'ckpt_dir': CKPT_DIR,
           'local_scratch': LOCAL, 'gates': GATES, 'ckpt_every': CKPT_EVERY},
          open(f'{RUN_DIR}/meta/run_config.json', 'w'), indent=2)
print('checkpoints ->', CKPT_DIR)
print('tensorboard ->', CKPT_DIR + '/tb')
print('gate reports->', RUN_DIR + '/gates | /attribution | /gradient | /summary')

## 8. Resume / checkpoint inspection
What IS restored on resume (`crl.train` semantics): env-step count, policy +
critic + target params, all optimizer states, adaptive-alpha state.
What is NOT restored (documented trainer limitation): the replay buffer
(refills from fresh collection; learning pauses for `min_replay_size` steps
after each resume), collection RNG streams, and the live env state.

In [ ]:
# 8. Inspect Drive checkpoints and enforce resume safety.
import pickle, glob
sys.path.insert(0, REPO)
import crl.losses  # TrainingState must be importable to unpickle

def ckpt_step(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)['step']
    except Exception:
        return None

latest = f'{CKPT_DIR}/latest.pkl'
found = ckpt_step(latest) if os.path.exists(latest) else None
print('latest.pkl step:', found)
for p in sorted(glob.glob(f'{RUN_DIR}/gates/gate_*.pkl')) + \
         sorted(glob.glob(f'{CKPT_DIR}/rolling_*.pkl')):
    print('  ', os.path.basename(p), 'step', ckpt_step(p))
if os.path.exists(f'{CKPT_DIR}/abort.pkl'):
    print('!! abort.pkl present (a numerical guard fired earlier) -- inspect '
          'metrics.json before resuming.')

if RESUME and found is None and not ALLOW_FRESH_START:
    raise RuntimeError('RESUME=True but no valid checkpoint found. Set '
                       'ALLOW_FRESH_START=True to explicitly start from 0.')
if (not RESUME) and found is not None and not ALLOW_FRESH_START:
    raise RuntimeError(f'A checkpoint at step {found} already exists. Set '
                       'RESUME=True to continue it, or ALLOW_FRESH_START=True '
                       '(with a new TAG/RUN_ID) to start over deliberately.')
print('resume check OK -> will', 'RESUME from', found) if RESUME else \
      print('resume check OK -> fresh start authorized')

## 9. Smoke tests -- the long run does not start unless ALL pass

In [ ]:
# 9. Full pre-training gates + checkpoint roundtrip + Drive write test.
os.chdir(REPO)
r = subprocess.run([sys.executable, 'scripts/verify_d4rl_ant.py'],
                   capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-800:])
assert 'ALL GATES PASS' in r.stdout, 'verify_d4rl_ant gates FAILED -- stopping.'

from crl.config import Config
from crl import envs as envs_mod, checkpoint as ckpt_mod
import numpy as np
cfg = Config(env_name=ENV)
env = envs_mod.make_env(ENV, cfg, seed=1)
obs = env.reset()
assert obs.shape == (58,) and cfg.goal_dim == 29 and cfg.obs_dim == 29
for _ in range(5):
    obs, rew, _, _ = env.step(np.random.uniform(-1, 1, 8).astype(np.float32))
    assert np.all(np.isfinite(obs)) and np.isfinite(rew)
print('env smoke OK: obs 58 (29 state + 29 goal), finite obs/rewards')

import crl.losses as losses_mod, jax
st = losses_mod.TrainingState(None, None, {'w': np.ones(3)}, {'w': np.ones(2)},
                              None, jax.random.PRNGKey(0))
ckpt_mod.save_named(LOCAL, 'smoke', 123, st)
s2, st2 = ckpt_mod.load_checkpoint(f'{LOCAL}/smoke.pkl')
assert s2 == 123
with open(f'{RUN_DIR}/meta/drive_write_test.txt', 'w') as f:
    f.write(time.strftime('%F %T'))
print('checkpoint roundtrip OK; Drive write OK. SMOKE TESTS PASS.')

## 10. Launch staged training (50k stages to 250k)
Each stage: `train(cfg)` (the repo's real entry point) to the next gate,
resuming from `latest.pkl`; then a validated copy of `latest.pkl` becomes the
mandatory `gate_<step>.pkl`. `latest.pkl` is written atomically every 10k
steps; a background thread keeps the 3 newest `rolling_*.pkl` copies. Full
stdout/stderr is teed to `logs/`. Guards (`guard_abort`) save `abort.pkl` and
stop the stage. Optional: run the next cell to watch TensorBoard first.

In [ ]:
# 10a. (optional) TensorBoard on the Drive run dir.
tb_logdir = CKPT_DIR + '/tb'
%load_ext tensorboard
%tensorboard --logdir $tb_logdir

In [ ]:
# 10b. Staged training loop. Rerunning this cell SKIPS finished stages.
import shutil, threading
from crl.config import Config
from crl.train import train

def build_cfg(stage_end, resume):
    cfg = Config(
        env_name=ENV, use_td=False, twin_q=False,        # binary NCE
        random_goals=0.5,
        entropy_coefficient=ENTROPY_COEF, target_entropy=TARGET_ENTROPY,
        guard_abort=True,
        max_number_of_steps=stage_end,
        min_replay_size=10_000, random_steps=10_000,
        num_sgd_steps_per_step=4, batch_size=256,
        eval_every_steps=CKPT_EVERY, eval_episodes=10, log_every_steps=5_000,
        seed=SEED, ckpt_dir=CKPT_DIR, resume=resume, tensorboard=True)
    assert cfg.use_td is False and cfg.twin_q is False, 'must be binary NCE'
    assert cfg.entropy_coefficient is None and cfg.target_entropy == -8.0
    assert cfg.guard_abort, 'numerical guards must stay on'
    assert cfg.max_number_of_steps <= MAX_STEP, 'no auto-continue past 250k'
    return cfg

class Tee:
    def __init__(self, path, stream):
        self.f, self.s = open(path, 'a', buffering=1), stream
    def write(self, x):
        self.f.write(x); self.s.write(x)
    def flush(self):
        self.f.flush(); self.s.flush()

def rolling_keeper(stop):
    while not stop.is_set():
        s = ckpt_step(f'{CKPT_DIR}/latest.pkl') if os.path.exists(
            f'{CKPT_DIR}/latest.pkl') else None
        if s is not None:
            dst = f'{CKPT_DIR}/rolling_{s}.pkl'
            if not os.path.exists(dst):
                shutil.copy2(f'{CKPT_DIR}/latest.pkl', dst + '.tmp')
                if ckpt_step(dst + '.tmp') == s:
                    os.replace(dst + '.tmp', dst)     # atomic, validated
            keep = sorted(glob.glob(f'{CKPT_DIR}/rolling_*.pkl'),
                          key=lambda p: int(p.split('_')[-1][:-4]))
            for p in keep[:-3]:
                os.remove(p)
        stop.wait(300)

for gate in GATES:
    ls = ckpt_step(f'{CKPT_DIR}/latest.pkl') if os.path.exists(
        f'{CKPT_DIR}/latest.pkl') else None
    gate_pkl = f'{RUN_DIR}/gates/gate_{gate}.pkl'
    if os.path.exists(gate_pkl):
        print(f'stage {gate}: gate checkpoint exists -- skip'); continue
    if ls is not None and ls >= gate:
        pass                                            # just snapshot below
    else:
        resume = ls is not None
        print(f'=== stage -> {gate} (resume={resume}, from step {ls}) ===')
        stop = threading.Event()
        th = threading.Thread(target=rolling_keeper, args=(stop,), daemon=True)
        th.start()
        out0, err0 = sys.stdout, sys.stderr
        sys.stdout = Tee(f'{RUN_DIR}/logs/train_to_{gate}.log', out0)
        sys.stderr = Tee(f'{RUN_DIR}/logs/train_to_{gate}.log', err0)
        try:
            train(build_cfg(gate, resume))
        finally:
            sys.stdout, sys.stderr = out0, err0
            stop.set(); th.join(timeout=5)
    # mandatory validated gate snapshot (tmp -> validate -> rename)
    shutil.copy2(f'{CKPT_DIR}/latest.pkl', gate_pkl + '.tmp')
    s = ckpt_step(gate_pkl + '.tmp')
    assert s is not None and s >= gate, f'gate snapshot invalid (step {s})'
    os.replace(gate_pkl + '.tmp', gate_pkl)
    print(f'gate_{gate}.pkl saved (step {s})')
    if os.path.exists(f'{CKPT_DIR}/abort.pkl'):
        print('!! GUARD FIRED (abort.pkl saved) -- stopping the staged loop.')
        break
print('staged training done at step',
      ckpt_step(f'{CKPT_DIR}/latest.pkl'))

## 11. Gate reports (JSON + Markdown, synced to Drive)

In [ ]:
# 11. Run the existing gate tools for every captured gate checkpoint.
os.chdir(REPO)
sys.path.insert(0, f'{REPO}/scripts')

def run_tool(args):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    if r.returncode != 0:
        print('TOOL FAILED:', args, '\n', r.stderr[-800:])
    return r.returncode == 0

def gate_summary(gate):
    tag = f'd4rl250k_{gate}'
    gpk = f'{RUN_DIR}/gates/gate_{gate}.pkl'
    out_json = f'{RUN_DIR}/summary/gate_{gate}.json'
    if not os.path.exists(gpk) or os.path.exists(out_json):
        return
    run_tool(['scripts/qualify_gate_report.py', '--ckpt', gpk,
              '--out', f'{RUN_DIR}/gates', '--tag', tag,
              '--target_entropy', str(TARGET_ENTROPY), '--env_name', ENV])
    run_tool(['scripts/goal_block_attribution.py', '--ckpt', gpk,
              '--env_name', ENV, '--tag', tag,
              '--out', f'{RUN_DIR}/attribution'])
    S = json.load(open(f'{RUN_DIR}/gates/gate_{tag}.json'))
    try:
        A = json.load(open(f'{RUN_DIR}/attribution/attr_{tag}.json'))
    except FileNotFoundError:
        A = None
    grad = None
    refs = f'{RUN_DIR}/gates/refs_{tag}.npz'
    if os.path.exists(refs):                # coverage-gated fresh states
        try:
            import numpy as np
            from ant_actor_gradient_audit import audit_arm
            grad = audit_arm(tag, ENV, gpk, refs, np.random.default_rng(0))
            json.dump(grad, open(f'{RUN_DIR}/gradient/grad_{tag}.json', 'w'),
                      indent=2)
        except Exception as e:              # report, never hide
            grad = {'error': str(e)}
    mets = json.load(open(f'{CKPT_DIR}/metrics.json'))
    near = min(mets, key=lambda e: abs(e['step'] - S['step']))
    combined = {'gate': gate, 'step': S['step'],
                'policy_head': S['policy_head'],
                'collection': S['collection'],
                'eval_deterministic': S['eval_deterministic'],
                'coverage_gates': S['coverage_gates'],
                'controllability': S['controllability'],
                'training_eval_nearest': near,
                'retrieval': None if A is None else
                    {'baseline_acc': A['baseline_acc'],
                     'rank_median': A['positive_rank_median'],
                     'blocks': A['blocks']},
                'actor_gradient': None if grad is None else
                    {k: grad[k] for k in ('verdict', 'checks', 'gradient')
                     if k in grad} | ({'error': grad['error']}
                                      if 'error' in grad else {})}
    json.dump(combined, open(out_json, 'w'), indent=2)
    h, e = S['policy_head'], S['eval_deterministic']
    L = [f"# Gate {gate} (step {S['step']})",
         f"alpha {h['alpha']} | entropy med {h['entropy_median']:.2f} | "
         f"scale {h['scale_median']:.3f} | floor {h['frac_at_min_std']:.2f} | "
         f"sat {h['mode_sat']:.2f}",
         f"collection: eff-rank {S['collection']['action_eff_rank']:.1f}, "
         f"moving {S['collection']['moving_transition_frac']:.2f}, "
         f"cells {S['collection']['unique_cells']}",
         f"eval: success {e['success']:.2f}, goal_vel {e['goal_vel']:.4f}, "
         f"dmin {e['dmin']:.2f}, speed {e['speed']:.4f}, "
         f"fall {e['fall_frac']:.2f}",
         f"training eval @{near['step']}: min_dist "
         f"{near.get('min_dist')}, final_dist {near.get('final_dist')}, "
         f"logits_gap {near.get('logits_gap')}",
         f"retrieval: {combined['retrieval'] and combined['retrieval']['baseline_acc']}",
         f"actor-gradient: "
         f"{combined['actor_gradient'] and combined['actor_gradient'].get('verdict')}"]
    open(f'{RUN_DIR}/summary/gate_{gate}.md', 'w').write('\n\n'.join(L) + '\n')
    print(open(f'{RUN_DIR}/summary/gate_{gate}.md').read())

for g in GATES:
    gate_summary(g)
print('gate reports done:',
      sorted(os.path.basename(p)
             for p in glob.glob(f'{RUN_DIR}/summary/gate_*.json')))

## 12. Resume instructions (after a Colab disconnect)
1. Reopen this notebook; `Runtime > Disconnect and delete runtime` if the old
   one is wedged.
2. Rerun sections **1 -> 9 in order** (mount Drive, clone, install, verify,
   smoke) with **`RESUME=True`** in section 1.
3. Section 8 prints the detected checkpoint step -- confirm it matches your
   expectation before continuing (it refuses to run otherwise).
4. Rerun the **section 10b training cell**: finished stages are skipped and
   training continues from `latest.pkl` (note: the replay buffer refills;
   learning pauses for the first 10k resumed steps).
5. Rerun sections 11/13/14 at any time -- they only read saved files.

In [ ]:
# 12b. Inspect the latest saved metrics WITHOUT touching training.
mets = json.load(open(f'{CKPT_DIR}/metrics.json'))
for e in mets[-6:]:
    print({k: (round(v, 4) if isinstance(v, float) else v)
           for k, v in e.items()
           if k in ('step', 'success', 'min_dist', 'final_dist', 'alpha',
                    'actor_loss', 'logits_gap', 'categorical_accuracy',
                    'guard_abort')})
print('checkpoints:', sorted(os.path.basename(p) for p in
      glob.glob(f'{CKPT_DIR}/*.pkl')))
print('gates      :', sorted(os.path.basename(p) for p in
      glob.glob(f'{RUN_DIR}/gates/gate_*.pkl')))

## 13. Final 250k summary + stop criteria (no auto-continue)

In [ ]:
# 13. Trend analysis across the 5 gates -> final verdict + next step.
gs = []
for g in GATES:
    p = f'{RUN_DIR}/summary/gate_{g}.json'
    if os.path.exists(p):
        gs.append(json.load(open(p)))
assert gs, 'no gate summaries yet -- run section 11 first'

def series(fn):
    return [fn(x) for x in gs]
gv   = series(lambda x: x['eval_deterministic']['goal_vel'])
dmin = series(lambda x: x['eval_deterministic']['dmin'])
fin  = series(lambda x: x['training_eval_nearest'].get('final_dist'))
spd  = series(lambda x: x['eval_deterministic']['speed'])
use  = series(lambda x: (x['controllability'] or {}).get('sigma_0.05', {})
              .get('decile_usefulness'))
suc  = series(lambda x: x['eval_deterministic']['success'])

def rising(v):
    v = [x for x in v if x is not None]
    return len(v) >= 3 and v[-1] > v[0] and v[-1] > 0
def falling(v):
    v = [x for x in v if x is not None]
    return len(v) >= 3 and v[-1] < v[0]

trends = {
    'positive_rising_goal_vel': rising(gv) and min(gv[-2:]) > 0.01,
    'decreasing_min_final_dist': falling(dmin) and falling(fin),
    'increasing_det_displacement': rising(spd),
    'improving_ctrl_usefulness': rising(use) and (use[-1] or 0) > 0.3,
    'nontrivial_success': all(s >= 0.1 for s in suc[-2:]),
}
any_trend = any(trends.values())
final = {'gates_analyzed': [x['gate'] for x in gs],
         'goal_vel': gv, 'dmin': dmin, 'final_dist': fin, 'speed': spd,
         'ctrl_usefulness': use, 'success': suc, 'trends': trends,
         'sustained_positive_trend': any_trend,
         'decision': ('CONTINUATION_CANDIDATE -- discuss before any run past '
                      '250k' if any_trend else
                      'NO_TREND -- next experiment should be a MATCHED-STEP '
                      'single-actor vs four-actor collection comparison, NOT '
                      'an immediate scale-up to 1M.')}
json.dump(final, open(f'{RUN_DIR}/summary/final_250k_summary.json', 'w'),
          indent=2)
open(f'{RUN_DIR}/summary/final_250k_summary.md', 'w').write(
    '# Final 250k summary\n\n' + json.dumps(final, indent=2))
print(json.dumps(final, indent=1))
print('\nSTOP: this notebook does not continue past 250k automatically.')

## 14. Package and sync artifacts

In [ ]:
# 14. Zip the small artifacts (reports/logs/meta, NOT checkpoints).
import shutil
pkg = f'{LOCAL}/artifacts_{RUN_ID}'
os.makedirs(pkg, exist_ok=True)
for d in ('gates', 'attribution', 'gradient', 'summary', 'meta', 'logs'):
    src = f'{RUN_DIR}/{d}'
    if os.path.exists(src):
        shutil.copytree(src, f'{pkg}/{d}', dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pkl'))
shutil.copy2(f'{CKPT_DIR}/metrics.json', f'{pkg}/metrics.json')
zp = shutil.make_archive(f'{LOCAL}/artifacts_{RUN_ID}', 'zip', pkg)
shutil.copy2(zp, f'{RUN_DIR}/summary/')
print('packaged ->', f'{RUN_DIR}/summary/{os.path.basename(zp)}')
!du -sh $RUN_DIR/* | sort -h